In [4]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [5]:
TRAIN_FILE = "taskA_train.csv"
HIDDEN_FILE = "taskA_hidden_public.csv"
OUTPUT_FILE = "submission_taskA.csv"

In [6]:
train = pd.read_csv(TRAIN_FILE)
hidden = pd.read_csv(HIDDEN_FILE)

train["timestamp"] = pd.to_datetime(train["timestamp"])
hidden["timestamp"] = pd.to_datetime(hidden["timestamp"])

train["text"] = train["text"].fillna("").astype(str)
hidden["text"] = hidden["text"].fillna("").astype(str)

In [7]:
all_text = pd.concat(
    [train["text"], hidden["text"]],
    ignore_index=True
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1
)

vectorizer.fit(all_text)

train_vectors = vectorizer.transform(train["text"])


In [8]:
def make_features(current, candidate, current_pos, candidate_pos,
                  current_vector, candidate_vector):

    # Text similarity
    similarity = float(
        current_vector.multiply(candidate_vector).sum()
    )

    # Word overlap
    words1 = set(current["text"].lower().split())
    words2 = set(candidate["text"].lower().split())

    overlap = len(words1 & words2) / max(
        1, len(words1 | words2)
    )

    # Time difference in hours
    time_gap = (
        current["timestamp"] - candidate["timestamp"]
    ).total_seconds() / 3600.0

    # Distance in message order
    position_gap = current_pos - candidate_pos

    # Same sender should normally NOT be a reply
    same_sender = int(
        current["sender"] == candidate["sender"]
    )

    return [
        similarity,
        overlap,
        np.log1p(max(time_gap, 0)),
        time_gap,
        position_gap,
        same_sender
    ]


In [9]:
X = []
y = []

for episode_id, group in train.groupby("episode_id", sort=False):

    group = group.reset_index(drop=True)

    for current_pos in range(len(group)):

        current = group.iloc[current_pos]
        current_vector = train_vectors[
            train.index[
                train["message_id"] == current["message_id"]
            ][0]
        ]

        # A message can only reply to an earlier message
        for candidate_pos in range(current_pos):

            candidate = group.iloc[candidate_pos]

            candidate_vector = train_vectors[
                train.index[
                    train["message_id"] == candidate["message_id"]
                ][0]
            ]

            features = make_features(
                current,
                candidate,
                current_pos,
                candidate_pos,
                current_vector,
                candidate_vector
            )

            X.append(features)

            # Positive if this candidate is the true reply
            y.append(
                int(
                    candidate["message_id"]
                    == current["reply_to_message_id"]
                )
            )


X = np.asarray(X)
y = np.asarray(y)






In [10]:
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X, y)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [11]:
predictions = []

for episode_id, group in hidden.groupby("episode_id", sort=False):

    group = group.reset_index(drop=True)

    # Get TF-IDF vectors for this episode
    episode_vectors = vectorizer.transform(group["text"])

    for current_pos in range(len(group)):

        current = group.iloc[current_pos]
        current_vector = episode_vectors[current_pos]

        # First message cannot reply to anything
        if current_pos == 0:
            predictions.append({
                "episode_id": episode_id,
                "message_id": current["message_id"],
                "reply_to_message_id": "NONE"
            })
            continue

        candidates = []
        feature_rows = []

        for candidate_pos in range(current_pos):

            candidate = group.iloc[candidate_pos]
            candidate_vector = episode_vectors[candidate_pos]

            features = make_features(
                current,
                candidate,
                current_pos,
                candidate_pos,
                current_vector,
                candidate_vector
            )

            candidates.append(candidate)
            feature_rows.append(features)

        feature_rows = np.asarray(feature_rows)

        probabilities = model.predict_proba(
            feature_rows
        )[:, 1]

        best_index = int(np.argmax(probabilities))
        best_probability = probabilities[best_index]

        best_candidate = candidates[best_index]

        # Confidence threshold.
        # Lower confidence means the message starts a new thread.
        if best_probability < 0.70:
            reply_to = "NONE"
        else:
            reply_to = best_candidate["message_id"]

        predictions.append({
            "episode_id": episode_id,
            "message_id": current["message_id"],
            "reply_to_message_id": reply_to
        })






In [12]:
prediction_df = pd.DataFrame(predictions)

hidden_order = hidden[
    ["episode_id", "message_id", "timestamp"]
].copy()

hidden_order = hidden_order.sort_values(
    ["episode_id", "timestamp"]
)

# Map message -> predicted parent
parent = dict(
    zip(
        prediction_df["message_id"],
        prediction_df["reply_to_message_id"]
    )
)


def find_root(message_id, episode_messages):
    """
    Follow reply links backwards until reaching NONE.
    """

    visited = set()
    current = message_id

    while True:

        if current in visited:
            # Safety against unexpected cycles
            return message_id

        visited.add(current)

        previous = parent.get(current, "NONE")

        if previous == "NONE":
            return current

        if previous not in episode_messages:
            return current

        current = previous


thread_rows = []

for episode_id, group in hidden_order.groupby(
    "episode_id",
    sort=False
):

    episode_messages = set(group["message_id"])

    roots = {}

    # Find root for every message
    for message_id in group["message_id"]:

        root = find_root(
            message_id,
            episode_messages
        )

        roots[message_id] = root

    # Sort roots by their first appearance
    root_order = []

    for message_id in group["message_id"]:

        root = roots[message_id]

        if root not in root_order:
            root_order.append(root)

    # Root -> T1, T2, T3...
    root_to_thread = {
        root: f"{episode_id}-T{i + 1}"
        for i, root in enumerate(root_order)
    }

    for message_id in group["message_id"]:

        thread_rows.append({
            "episode_id": episode_id,
            "message_id": message_id,
            "thread_id": root_to_thread[roots[message_id]]
        })


thread_df = pd.DataFrame(thread_rows)


In [13]:
submission = prediction_df.merge(
    thread_df,
    on=["episode_id", "message_id"],
    how="left"
)

submission = submission[
    [
        "episode_id",
        "message_id",
        "reply_to_message_id",
        "thread_id"
    ]
]


In [14]:
# Correct number of rows
assert len(submission) == len(hidden)

# No duplicate message rows
assert not submission.duplicated(
    ["episode_id", "message_id"]
).any()

# Every hidden message appears exactly once
assert set(submission["message_id"]) == set(
    hidden["message_id"]
)

# Every reply must be NONE or an earlier message
message_time = dict(
    zip(hidden["message_id"], hidden["timestamp"])
)

for _, row in submission.iterrows():

    reply = row["reply_to_message_id"]

    if reply != "NONE":

        assert reply in message_time

        assert (
            message_time[reply]
            < message_time[row["message_id"]]
        )


In [15]:
submission.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Done!")
print(f"Output saved to: {OUTPUT_FILE}")
print()
print(submission.head(20).to_string(index=False))

Done!
Output saved to: submission_taskA.csv

episode_id message_id reply_to_message_id thread_id
      A002  A002-M001                NONE   A002-T1
      A002  A002-M005           A002-M001   A002-T1
      A002  A002-M002           A002-M005   A002-T1
      A002  A002-M003           A002-M002   A002-T1
      A002  A002-M008           A002-M003   A002-T1
      A002  A002-M009           A002-M008   A002-T1
      A002  A002-M004           A002-M009   A002-T1
      A002  A002-M010                NONE   A002-T2
      A002  A002-M006                NONE   A002-T3
      A002  A002-M007           A002-M006   A002-T3
      A003  A003-M001                NONE   A003-T1
      A003  A003-M002           A003-M001   A003-T1
      A003  A003-M004           A003-M002   A003-T1
      A003  A003-M005           A003-M004   A003-T1
      A003  A003-M006           A003-M005   A003-T1
      A003  A003-M008           A003-M006   A003-T1
      A003  A003-M009           A003-M008   A003-T1
      A003  A003-M0